# Lab Day 19: GraphRAG với Tech Company Corpus

Notebook này chạy pipeline từ indexing đến benchmark cho corpus trong `data/corpus.json`.

## 1. Thiết lập môi trường

Cài dependencies nếu cần:

```python
%pip install -r ../requirements.txt
```

Điền `OPENAI_API_KEY` trong file `.env` ở repo root. Nếu chưa điền key, notebook vẫn chạy được bằng fallback offline.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.graphrag_lab import Config, load_corpus, chunk_documents
from src.graphrag_lab import index_pipeline, load_or_build_artifacts
from src.graphrag_lab import ask_flat, ask_graph, run_benchmark

cfg = Config.from_env(ROOT / '.env')
cfg.data_path = ROOT / cfg.data_path
cfg.artifact_dir = ROOT / cfg.artifact_dir
cfg

## 2. Khám phá corpus

In [ ]:
docs = load_corpus(cfg.data_path)
[(doc.id, doc.title, len(doc.content)) for doc in docs]

In [ ]:
chunks = chunk_documents(docs, cfg.chunk_size_words, cfg.chunk_overlap_words, cfg.max_chunks_per_doc)
len(chunks), chunks[0].id, chunks[0].title, chunks[0].text[:500]

## 3. Indexing và Graph Construction

In [ ]:
metrics = index_pipeline(cfg)
metrics

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(cfg.artifact_dir / 'knowledge_graph.png')))

## 4. Querying: Flat RAG vs GraphRAG

In [ ]:
chunks, triples, graph = load_or_build_artifacts(cfg)
question = 'What is the relationship between OpenAI and Microsoft?'
flat = ask_flat(question, chunks, cfg)
graph_answer = ask_graph(question, graph, chunks, cfg)
flat['answer'], graph_answer['answer']

In [ ]:
import pandas as pd
pd.DataFrame([flat, graph_answer])[['mode', 'seconds', 'total_tokens', 'context_chars', 'answer']]

## 5. Benchmark 20 câu hỏi

In [ ]:
rows = run_benchmark(cfg)
bench = pd.DataFrame(rows)
bench[['question', 'mode', 'seconds', 'total_tokens', 'context_chars', 'answer']].head(10)

In [ ]:
bench.groupby('mode')[['seconds', 'total_tokens', 'context_chars']].mean().round(2)

## 6. Phân tích ngắn

- Flat RAG phù hợp với câu hỏi đơn hop, nơi câu trả lời nằm trong một chunk gần keyword.
- GraphRAG phù hợp với câu hỏi cần nối nhiều entity/relation hoặc multi-hop.
- Khi chấm benchmark, xem cột `answer` để ghi lại trường hợp Flat RAG bị hallucination hoặc lấy sai context, trong khi GraphRAG đúng hơn nhờ facts có cấu trúc.